In [4]:
#!gsutil cp -r gs://bucket-for-kpf-project/UCI_Credit_Card.csv ./

In [2]:
!pip install --quiet --upgrade "kfp>=2,<3"
!pip install --quiet google.cloud
!pip install --quiet google-cloud-aiplatform
!pip install -U --quiet google-cloud-pipeline-components

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/opt/micromamba/lib/python3.12/site-packages/kfp-2.17.0.dist-info/INSTALLER0lm2amvo.tmp'



In [4]:
import kfp
from google.cloud import aiplatform
from google_cloud_pipeline_components.v1.dataset import TabularDatasetCreateOp
from google_cloud_pipeline_components.v1.custom_job import \
    CustomTrainingJobOp
from google_cloud_pipeline_components.v1.endpoint.create_endpoint.component import \
    endpoint_create as EndpointCreateOp
from google_cloud_pipeline_components.v1.endpoint.deploy_model.component import \
    model_deploy as ModelDeployOp
from kfp.dsl import Output, Model
from google_cloud_pipeline_components._implementation.model import GetVertexModelOp

In [12]:
PROJECT_ID = "project-c8d59959-80e8-496f-96a"
BUCKET_URI = "gs://custom-trin-bucket"
DATASET_URI = "gs://bucket-for-kpf-project/UCI_Credit_Card.csv"
GCP_REGION = "us-central1"
DISPLAY_NAME = "custom-credit-card-10Jul26-1"
MACHINE_TYPE = "n1-standard-4"
TRAIN_URI = "us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest"
SERVING_URI = "us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest"
DATASET_URI = 'projects/90514214177/locations/us-central1/datasets/7580367688300494848'

In [15]:
def create_training_pipeline_custom_job_sample(
    replica_count: int = 1,
    machine_type: str = "MACHINE_TYPE",
    accelerator_type: str = "ACCELERATOR_TYPE_UNSPECIFIED",
    accelerator_count: int = 0,
    training_fraction_split: float = 0.8,
    validation_fraction_split: float = 0.1,
    test_fraction_split: float = 0.1,
    sync: bool = True,
):
    aiplatform.init(project=PROJECT_ID, location=GCP_REGION, staging_bucket=BUCKET_URI)

    job = aiplatform.CustomTrainingJob(
        display_name=DISPLAY_NAME,
        script_path="task.py",
        container_uri=TRAIN_URI,
        model_serving_container_image_uri=SERVING_URI,
    )

    dataset = aiplatform.TabularDataset(DATASET_URI)

    model = job.run(
        dataset=dataset,
        model_display_name=DISPLAY_NAME,
        #args=args,
        replica_count=replica_count,
        machine_type=MACHINE_TYPE,
        training_fraction_split=training_fraction_split,
        validation_fraction_split=validation_fraction_split,
        test_fraction_split=test_fraction_split,
        sync=sync,
    )

    model.wait()

    print(model.display_name)
    print(model.resource_name)
    print(model.uri)
    return model, model.resource_name



In [16]:
traied_model, trained_model_name = create_training_pipeline_custom_job_sample()

Training script copied to:
gs://custom-trin-bucket/aiplatform-2026-08-21-14:20:41.628-aiplatform_custom_trainer_script-0.1.tar.gz.
Training Output directory:
gs://custom-trin-bucket/aiplatform-custom-training-2026-08-21-14:20:41.851 
View Training:
https://console.cloud.google.com/agent-platform/locations/us-central1/training/1609001272860475392?project=90514214177
CustomTrainingJob projects/90514214177/locations/us-central1/trainingPipelines/1609001272860475392 current state:
PIPELINE_STATE_RUNNING
CustomTrainingJob projects/90514214177/locations/us-central1/trainingPipelines/1609001272860475392 current state:
PIPELINE_STATE_RUNNING
CustomTrainingJob projects/90514214177/locations/us-central1/trainingPipelines/1609001272860475392 current state:
PIPELINE_STATE_RUNNING
CustomTrainingJob projects/90514214177/locations/us-central1/trainingPipelines/1609001272860475392 current state:
PIPELINE_STATE_RUNNING
CustomTrainingJob projects/90514214177/locations/us-central1/trainingPipelines/16090

In [18]:
@kfp.dsl.pipeline(
    name=DISPLAY_NAME,
    pipeline_root=BUCKET_URI,
    
    )
def pipeline(PROJECT_ID: str):

    model_task = GetVertexModelOp(
        model_name=trained_model_name,
    )
    endpoint_op = EndpointCreateOp(
        project=PROJECT_ID,
        location=GCP_REGION,
        display_name=DISPLAY_NAME,
    )

    ModelDeployOp(
        model=model_task.outputs["model"],
        endpoint=endpoint_op.outputs["endpoint"],
        dedicated_resources_min_replica_count=1,
        dedicated_resources_max_replica_count=1,
        dedicated_resources_machine_type=MACHINE_TYPE,
    )

In [19]:
kfp.compiler.Compiler().compile(
    pipeline_func=pipeline,
    package_path='custom_classif_pipeline_10jul_v1.yaml'
)
!gsutil cp -r custom_classif_pipeline_10jul_v1.yaml gs://custom-trin-bucket/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://custom_classif_pipeline_10jul_v1.yaml [Content-Type=application/yaml]...
/ [1 files][ 18.1 KiB/ 18.1 KiB]                                                
Operation completed over 1 objects/18.1 KiB.                                     


In [21]:
aiplatform.init(
    project=PROJECT_ID,
    location=GCP_REGION,
)

# Prepare the pipeline job
job = aiplatform.PipelineJob(
    display_name=DISPLAY_NAME,
    template_path="custom_classif_pipeline_10jul_v1.yaml",
    pipeline_root=BUCKET_URI,
    parameter_values={
        "PROJECT_ID": PROJECT_ID,
    },
    enable_caching=True,
)

job.submit()

Creating PipelineJob
PipelineJob created. Resource name: projects/90514214177/locations/us-central1/pipelineJobs/custom-credit-card-10jul26-1-20260821145643
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/90514214177/locations/us-central1/pipelineJobs/custom-credit-card-10jul26-1-20260821145643')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/custom-credit-card-10jul26-1-20260821145643?project=90514214177


In [22]:
endpoint = aiplatform.Endpoint(
    endpoint_name = "projects/90514214177/locations/us-central1/endpoints/6371578073741000704",
)

In [25]:
endpoint.predict(instances=[[19503, 50000, 1, 1, 1, 57, -1, 1, -1, 0, 0, 0,
                            8617, 5670, 35835, 20940, 19146, 19131,
                            2000, 36681, 10000, 9000, 689, 679],
                            [308, 50000, 1, 1, 1, 23, -1, 1, -1, 0, 0, 0,
                            8617, 567000, 35835, 20940, 19146, 19131,
                            0, 0, 0, 0, 0, 0]])

Prediction(predictions=[[0.0], [1.0]], deployed_model_id='3887957228455460864', metadata=None, model_version_id='1', model_resource_name='projects/90514214177/locations/us-central1/models/6874446313774120960', explanations=None)